# Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 1 (2026–2027) – Major Project**

This notebook analyzes the supplied agricultural dataset to identify seasonal patterns, trends, relationships and differences in agricultural performance.

### Project workflow
1. Load and understand the dataset
2. Clean and prepare the data
3. Perform exploratory data analysis
4. Compare performance across seasons
5. Analyze crops, irrigation methods and regions
6. Study relationships between environmental/resource variables and outcomes
7. Apply a non-parametric statistical test for seasonal differences
8. Draw evidence-based conclusions and recommendations


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load the dataset

The next cell works both in Google Colab and in a normal Jupyter environment. If the CSV is not found, Colab will open an upload dialog.


In [ ]:
# Flexible file loading for Google Colab / Jupyter
candidate_files = [
    "seasonal_agriculture_performance_dataset (3).csv",
    "seasonal_agriculture_performance_dataset.csv",
    "/content/seasonal_agriculture_performance_dataset (3).csv"
]

csv_file = next((f for f in candidate_files if os.path.exists(f)), None)

if csv_file is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_file = next(iter(uploaded.keys()))
    except Exception as e:
        raise FileNotFoundError(
            "CSV not found. Upload the supplied dataset CSV to the notebook and run this cell again."
        )

df = pd.read_csv(csv_file)
print("Loaded:", csv_file)
print("Shape:", df.shape)
df.head()


## 2. Dataset structure and data quality

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing"))

print("\nDuplicate rows:", df.duplicated().sum())


### Cleaning approach

The supplied dataset contains a small number of missing numeric observations and no duplicate rows. Missing numeric values are replaced with the median of the corresponding column because the missing proportion is small and median imputation is less sensitive to extreme values than mean imputation. Categorical values are filled with the mode if needed.


In [ ]:
clean = df.copy()

numeric_cols = clean.select_dtypes(include=np.number).columns
categorical_cols = clean.select_dtypes(include="object").columns

for col in numeric_cols:
    clean[col] = clean[col].fillna(clean[col].median())

for col in categorical_cols:
    if clean[col].isna().any():
        clean[col] = clean[col].fillna(clean[col].mode().iloc[0])

print("Missing values after cleaning:", clean.isna().sum().sum())
print("Duplicates after cleaning:", clean.duplicated().sum())


## 3. Descriptive statistics

In [ ]:
display(clean.describe(include="all").T)


## 4. Seasonal performance analysis

Because production and profit are highly skewed in this dataset, both mean and median are useful. Medians are emphasized when comparing typical farm-level performance.


In [ ]:
season_summary = df.groupby("Season").agg(
    Farms=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Median_Yield=("Yield_Tonnes_Ha","median"),
    Avg_Production=("Production_Tonnes","mean"),
    Avg_Revenue=("Revenue_INR","mean"),
    Avg_Cost=("Total_Cost_INR","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Median_Profit=("Profit_INR","median"),
    Avg_Water=("Water_Used_m3","mean"),
    Avg_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Risk=("Disease_Pest_Risk_pct","mean")
).round(2)

display(season_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
season_summary["Median_Yield"].plot(kind="bar", ax=ax)
ax.set_title("Median Yield by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Yield (tonnes/ha)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
season_summary["Median_Profit"].plot(kind="bar", ax=ax)
ax.set_title("Median Profit by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Profit (INR per farm)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
season_summary["Avg_Risk"].plot(kind="bar", ax=ax)
ax.set_title("Average Disease/Pest Risk by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Risk (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Crop-level analysis

In [ ]:
crop_summary = clean.groupby("Crop").agg(
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Production=("Production_Tonnes","mean"),
    Avg_Revenue=("Revenue_INR","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Risk=("Disease_Pest_Risk_pct","mean")
).sort_values("Avg_Profit", ascending=False).round(2)

display(crop_summary)


In [ ]:
crop_summary["Avg_Profit"].plot(kind="bar", figsize=(8,4))
plt.title("Average Profit by Crop")
plt.xlabel("Crop")
plt.ylabel("Profit (INR per farm)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 6. Irrigation analysis

In [ ]:
irrigation_summary = clean.groupby("Irrigation_Method").agg(
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Water=("Water_Used_m3","mean"),
    Avg_Efficiency=("Water_Efficiency_t_per_1000m3","mean")
).sort_values("Avg_Profit", ascending=False).round(2)

display(irrigation_summary)


In [ ]:
irrigation_summary["Avg_Efficiency"].plot(kind="bar", figsize=(7,4))
plt.title("Water Efficiency by Irrigation Method")
plt.xlabel("Irrigation Method")
plt.ylabel("Tonnes per 1,000 m³")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 7. Regional analysis

In [ ]:
state_summary = clean.groupby("State").agg(
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Revenue=("Revenue_INR","mean"),
    Avg_Efficiency=("Water_Efficiency_t_per_1000m3","mean")
).sort_values("Avg_Profit", ascending=False).round(2)

display(state_summary)


## 8. Relationship analysis

Spearman correlation is used because several variables are strongly skewed and may not satisfy the assumptions of Pearson correlation. Correlation indicates association, not causation.


In [ ]:
key_vars = [
    "Rainfall_mm","Avg_Temperature_C","Humidity_pct","Sunlight_Hours_Day",
    "Soil_Moisture_pct","Seed_Quality_Score","Fertilizer_kg_ha",
    "Water_Used_m3","Market_Price_INR_Tonne","Yield_Tonnes_Ha","Profit_INR"
]

corr = clean[key_vars].corr(method="spearman")
display(corr.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(9,7))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(key_vars)))
ax.set_yticks(range(len(key_vars)))
ax.set_xticklabels(key_vars, rotation=55, ha="right", fontsize=7)
ax.set_yticklabels(key_vars, fontsize=7)
ax.set_title("Spearman Correlation Matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
# Selected Spearman relationships with yield
for col in [
    "Rainfall_mm","Avg_Temperature_C","Humidity_pct","Sunlight_Hours_Day",
    "Soil_Moisture_pct","Seed_Quality_Score","Fertilizer_kg_ha","Water_Used_m3",
    "Market_Price_INR_Tonne"
]:
    tmp = clean[[col, "Yield_Tonnes_Ha"]].dropna()
    r, p = spearmanr(tmp[col], tmp["Yield_Tonnes_Ha"])
    print(f"{col:30s} rho={r: .3f}, p={p:.3g}")


## 9. Statistical test for seasonal differences

A Kruskal–Wallis test is used to test whether the distribution of a performance variable differs across Kharif, Rabi and Zaid. A p-value below 0.05 is treated as evidence of a statistically significant difference among seasons.


In [ ]:
for metric in ["Yield_Tonnes_Ha", "Profit_INR", "Revenue_INR", "Disease_Pest_Risk_pct"]:
    groups = [g[metric].dropna().values for _, g in df.groupby("Season")]
    stat, p = kruskal(*groups)
    print(f"{metric:30s} H={stat:.3f}, p={p:.4g}")


## 10. Key findings

- Kharif has the highest median yield and median profit among the three seasons.
- Zaid has the lowest median yield and a negative median profit.
- Disease/pest risk is highest in Kharif and lowest in Zaid.
- Sugarcane and Chilli show the strongest average profitability in the supplied data.
- Drip irrigation has the highest average yield and average profit among the four irrigation methods, while Rainfed has the highest water-efficiency metric.
- Market price has a notable negative Spearman association with yield in this dataset; this should not be interpreted as a causal relationship.
- The Kruskal–Wallis results indicate statistically significant seasonal differences for yield, profit, revenue and disease/pest risk.


## 11. Recommendations

1. Use season-specific planning rather than treating all seasons as equivalent.
2. Investigate practices associated with the stronger Kharif outcomes.
3. Prioritize water-efficient practices while considering yield and profitability together.
4. Review the economics of crops/seasons with persistent negative median profit.
5. Monitor disease and pest risk closely during higher-risk periods.
6. Extend the analysis with more years, weather observations and market information before making operational decisions.


## 12. Conclusion

The analysis demonstrates clear seasonal variation in agricultural performance within the supplied dataset. Kharif generally shows stronger typical yield and profitability, whereas Zaid shows weaker economic outcomes. Crop choice, irrigation method, environmental conditions and market variables are associated with different performance measures. The findings support evidence-based seasonal planning, while also highlighting the need for more contextual data before drawing causal conclusions.
